# Rust 105: Ownership and Borrowing

Chapter 05 of the Rust track, and the one the whole language is organised around. Chapters 01–04 kept touching the same wall: `&String` versus `&str` in a signature, the live slice that blocked a `push`, the `E0106` behind returning a borrowed view of a local. This chapter is the wall. Ownership is the rule for **who is responsible for a value**, and borrowing is the rule for **who may look at it while they are not responsible**.

Everything here is a **runnable example**. Read the markdown, run the cell below it, and change nothing — the practice twin of this notebook is `05_ownership-and-borrowing_p.ipynb`.

## Who this is written for

You are assumed to be a competent programmer, arriving from Python and C++. Python decides lifetime by garbage collection; C++ decides it by whoever copies first, and `std::move` and dangling references are the price. Rust decides it at compile time, and the compiler tells you when the decision does not work. What is new is not the *idea* of ownership — every language has some answer to it — but the fact that it is a checked, per-value rule instead of a runtime mechanism.

## How to read this notebook

- Markdown cells explain; code cells demonstrate. **Run each code cell as you reach it.** Several sections build one running example across cells, so out-of-order runs will leave bindings undefined.
- Code that intentionally *fails* to compile is shown as a comment, together with the diagnostic you would get. In this chapter more than any other, those diagnostics are the material: learning Rust's ownership model largely means learning to read five error codes.
- The prose is terse and the facts are dense. This is a reference you will come back to, not a novel you read once.

## The one idea to internalise

Every value has exactly **one owner** at a time. Assignment, argument passing and returning **move** the owner unless the type is `Copy`; a **borrow** (`&T` or `&mut T`) is a temporary, checked loan rather than a second owner. The compiler enforces two rules everywhere, at compile time:

1. A value may be **moved** any number of times, but after each move the previous binding is gone.
2. At any moment there may be **either any number of shared borrows or exactly one mutable borrow — not both** — and no borrow may outlive its owner.

Everything else in the chapter is consequence, vocabulary, or the shape of the five diagnostics that report violations. When the compiler rejects a program here, it is not a style objection: it is reporting that two parts of the program disagree about who owns a value.


### Contents

- [0. Notebook Conventions: How These Cells Run](#0-notebook-conventions-how-these-cells-run)
- [1. Move Semantics: The Default](#1-move-semantics-the-default)
- [2. `Copy` Types: The Exception](#2-copy-types-the-exception)
- [3. Borrowing: `&T` and `&mut T`](#3-borrowing-t-and-mut-t)
- [4. The Borrow Rules, and NLL](#4-the-borrow-rules-and-nll)
- [5. Slices as Borrowed Views](#5-slices-as-borrowed-views)
- [6. `Box` and Moving Out](#6-box-and-moving-out)
- [7. Structs and Enums: Ownership in Composition](#7-structs-and-enums-ownership-in-composition)
- [8. Ownership in Function Signatures](#8-ownership-in-function-signatures)
- [9. Method Receivers: `&self`, `&mut self`, `self`](#9-method-receivers-self-mut-self-self)
- [10. Lifetimes and Elision](#10-lifetimes-and-elision)
- [11. `Option` in Depth](#11-option-in-depth)
- [12. The Borrow Checker in Real Loops](#12-the-borrow-checker-in-real-loops)
- [13. Reading the Compiler: The Five Codes](#13-reading-the-compiler-the-five-codes)
- [14. Performance Notes for Low-Latency Code](#14-performance-notes-for-low-latency-code)
- [15. Mental Model and Habits](#15-mental-model-and-habits)
- [16. Cheat Sheet](#16-cheat-sheet)

If you are returning to this chapter later, §13 is the fastest route back (the five diagnostics, each with its mechanical fix), and §16 is the one-page summary. §15 is the part to re-read when code keeps fighting you.


## 0. Notebook Conventions: How These Cells Run

This notebook runs on the **evcxr** kernel, which wraps each cell into a generated program and compiles it incrementally. Three consequences matter for every cell below:

- **There is no `fn main`.** The kernel supplies one; you are typing its body.
- **State persists across cells.** A `let` from cell 4 is still live in cell 30.
- **Items are order-independent, statements are not.** Two `fn`s may call each other regardless of order; two `let`s may not use each other's names before they run.

There is **one namespace per notebook**. Two items with the same name collide exactly as they would in one source file, even in different sections. If you re-run a cell that defines an item, it is redefined; on a redefinition error, restart the kernel and run from the top.

A failing cell does not poison the kernel — the deliberate compile errors in this chapter are comments, so every cell still runs. `println!` writes to stdout; `eprintln!` and `dbg!` write to stderr.

### A habit worth forming

In this chapter, more than anywhere else, the error message is the teaching material. When a commented-out example says it produces `E0382`, the best exercise is to *write the failing line yourself in a scratch cell*, read the real diagnostic, and only then read the fix. The compiler's hints are usually the fix, spelled out.


In [ ]:
// 0.1 Smoke test: one owner, moved once, the old binding is gone.
let owner = String::from("mine");
let moved = owner;
println!("moved = {moved}");


## 1. Move Semantics: The Default

Assigning a non-`Copy` value does not copy it — it **moves** the ownership from the old binding to the new one, in O(1), without touching the heap. The old binding is then out of scope in the strict sense: the compiler will refuse any further use of it.

| Operation | Moves? | Example |
|---|---|---|
| `let b = a;` | Yes, for non-`Copy` types | `let s2 = s1;` |
| passing to a function by value | Yes | `consume(s1)` |
| returning from a function | Yes | `fn make() -> String` |
| `if let` / `match` by value | Moves the matched parts | §7 |
| putting into a `Vec`/`HashMap` | Yes | `items.push(s1)` |

A move is *shallow*: the pointer, length and capacity are copied into the new binding, and the old one is marked dead. Nothing deep is duplicated — which is precisely why using the old binding afterwards would be a use-after-free, and why the compiler's refusal is not pedantry.


In [ ]:
// 1.1 Assignment moves: the old binding is dead immediately.
let s1 = String::from("hello");
let s2 = s1;
println!("s2 = {s2}");
// println!("{s1}");
//
// error[E0382]: borrow of moved value: `s1`
//  |
//  | let s1 = String::from("hello");
//  |         -- move occurs because `s1` has type `String`, which does not
//  |            implement the `Copy` trait
//  | let s2 = s1;
//  |            -- value moved here
//  | println!("{s1}");
//  |             ^^ value borrowed here after move
// help: consider cloning the value if the performance cost is acceptable
//  |
//  | let s2 = s1.clone();


### Moves happen at every assignment-shaped place

The rule is not "assignment moves" — it is "**every transfer of ownership moves**". That includes function arguments, returns, `Vec::push`, tuple and struct construction, and `Some(s1)`. Each of these takes the value by value, so each of them is a move of a non-`Copy` value.

The compiler tracks the *flow*, not the syntax. A value that was moved on one branch and not the other is only "maybe moved", and using it after the branch join is the same `E0382` — with a note saying the move was conditional.


In [ ]:
// 1.2 Conditional moves leave a maybe-moved value behind.
let token = String::from("abc");
let uppercase = true;
if uppercase {
    let raised = token;
    println!("raised = {raised}");
} else {
    println!("kept = {token}");
}
// println!("{token}");
//
// error[E0382]: borrow of moved value: `token`
//   = note: move occurs because `token` has type `String`, which does not
//           implement the `Copy` trait
//   = note: value moved into `raised` here, in the `true` branch


In [ ]:
// 1.3 Moves compose: into a Vec, into a tuple, into Some.
let first = String::from("a");
let second = String::from("b");
let mut items = Vec::new();
items.push(first);
let pair = (second, 2i32);
println!("items = {items:?}, pair = {pair:?}");
// first and second are both gone; items and pair own their payloads now.


### `mut` is a property of the binding, not the value

Re-read that sentence until it stops being obvious, because it is the key to half of all borrow errors. `let mut s = ...` makes *the binding* `s` mutable: you may reassign it, or borrow it mutably through it. The value itself has no mutability. A non-`mut` binding that owns a `String` cannot be pushed to; moving it into a `mut` binding transfers the value to a binding that can.

That is also why "make it `mut`" is sometimes the fix and sometimes not: if the value is behind a shared reference, no amount of `mut` on your binding helps, because the mutability that matters belongs to the binding the reference came from.


In [ ]:
// 1.4 Moving into a `mut` binding upgrades what you may do to the value.
let frozen = String::from("ice");
let mut thawed = frozen;
thawed.push_str(" -> water");
println!("{thawed}");
// frozen is gone; thawed is a different binding owning the same buffer.


In [ ]:
// 1.5 `mut` on a parameter is still just a local binding.
fn increment(mut value: i32) -> i32 {
    value += 1;
    value
}
let n = 5;
println!("increment({n}) = {}", increment(n));
println!("n is unchanged and still usable: {n}");


## 2. `Copy` Types: The Exception

Some types are so small and trivially re-duplicable that moving them would be pure overhead: integers, floats, `bool`, `char`, and any struct/enum/tuple/array composed exclusively of such types. These implement the `Copy` marker trait, and assignment **copies** instead of moving. The old binding stays alive.

| `Copy` | Not `Copy` |
|---|---|
| `i8`…`i128`, `u8`…`u128`, `isize`/`usize` | `String`, `Vec<T>`, `HashMap<K, V>` |
| `f32`, `f64` | `&mut T` |
| `bool`, `char` | any struct containing a non-`Copy` field |
| `&T` (shared references) | arrays/tuples containing non-`Copy` elements |
| arrays/tuples of `Copy` elements | |

Two consequences worth internalising now:

- **`Copy` is all-or-nothing for composites.** One `String` field makes the whole struct move-only.
- **Shared references are `Copy`, mutable ones are not.** `&T` copies freely; `&mut T` must be exclusive, so copying it would break the borrow rules by construction.

`Clone` is the explicit sibling: `.clone()` copies *any* cloneable type, including `String`, at the cost you would expect. `Copy` types also implement `Clone`, but the compiler never requires you to write `.clone()` for them.


In [ ]:
// 2.1 Copy types survive assignment; the composite of Copy elements is Copy too.
let n = 5i32;
let also_n = n;
println!("both alive: {n} and {also_n}");

let tuple: (i32, f64, char) = (1, 2.5, 'x');
let tuple_copy = tuple;
println!("tuple both alive: {tuple:?} and {tuple_copy:?}");

let bytes = [1u8, 2, 3];
let bytes_copy = bytes;
println!("array both alive: {bytes:?} and {bytes_copy:?}");


In [ ]:
// 2.2 One non-Copy field poisons the whole struct.
struct Price {
    symbol: String,   // not Copy
    ticks: u32,       // Copy
}
let p = Price { symbol: String::from("BTC"), ticks: 3 };
let referenced = &p;
println!("borrow fine: {} {}", referenced.symbol, referenced.ticks);
// let moved = p;
// println!("{}", p.ticks);
//
// error[E0382]: borrow of moved value: `p.ticks`
//   = note: move occurs because `p.ticks` has type `u32`, which does not
//           implement the `Copy` trait... wait, it does; the real note says
//           `p` is moved because `Price` does not implement `Copy`


In [ ]:
// 2.3 References are Copy when shared, exclusive when mutable — and that is the rules from §4 in one line.
let text = String::from("borrow me");
let a = &text;
let b = a;
println!("a = {a}, b = {b}, text = {text}");


## 3. Borrowing: `&T` and `&mut T`

A **borrow** is a temporary loan with a checked lifetime. `&T` is a **shared borrow**: any number may exist, none may mutate. `&mut T` is a **mutable borrow**: exactly one may exist, and it excludes all other borrows — shared or mutable — while it is alive. A borrow does not own; when the borrow ends, the value is exactly where it was.

The two borrow kinds are not "read and write" in a casual sense; they are two *capability levels*:

| Capability | `&T` | `&mut T` |
|---|---|---|
| read the value | yes | yes |
| mutate through the borrow | no | yes |
| have more than one alive at once | yes | no |
| coexist with the other kind | yes | no |
| outlive the owner | no | no |
| alias the same target | yes | no |

`&mut T` is deliberately heavier than a C++ non-const pointer: it is exclusive. That exclusivity is what makes the second rule of the chapter enforceable, and what lets the compiler cache-reorder around borrows without ever observing a surprise.


In [ ]:
// 3.1 Shared borrows are read-only views; you may have as many as you like.
let text = String::from("shared");
let r1 = &text;
let r2 = &text;
let r3 = &text;
println!("{r1} / {r2} / {r3} — length {}", r2.len());


In [ ]:
// 3.2 A mutable borrow is exclusive: one at a time, through and through.
//fn grow(text: &mut String) {
//    text.push('!');
//}
//let mut text = String::from("base");
//grow(&mut text);
//println!("{text}");


### Deref coercion hides one conversion you would otherwise write constantly

`&String` and `&str` are different types, but a function taking `&str` accepts `&String` without ceremony, because the compiler performs **deref coercion**: `&String` → `&str` via `String`'s `Deref` to `str`. The same mechanism accepts `&Vec<i32>` where `&[i32]` is wanted, and `&Box<T>` where `&T` is wanted.

Coercion only fires at coercion sites (arguments, `let` with a type annotation, returns) and only through `Deref`. It is why the convention "accept `&str`" costs the caller nothing — and why a `&String` parameter is almost always a mistake (chapter 04, §10).


In [ ]:
// 3.3 Deref coercion: &String becomes &str at the call site.
fn measure(text: &str) -> usize {
    text.len()
}
let owned = String::from("coerced");
println!("&String -> &str: {}", measure(&owned));
let literal: &str = "plain";
println!("&str: {}", measure(literal));
println!("slice of a String: {}", measure(&owned[0..4]));


### Function parameters borrow by convention

| Parameter | Caller keeps using it? | Callee may mutate? | Typical use |
|---|---|---|---|
| `T` | no — it was moved | it owns it | consuming, builders |
| `&T` | yes | no | read-only access |
| `&mut T` | after the call returns | yes, during | in-place update |

The convention from chapter 04 generalises: **accept the weakest borrow that does the job.** Read-only functions take `&T` (or `&str`/`&[T]`); functions that update in place take `&mut T`; functions that consume take `T` by value.


In [ ]:
// 3.4 The three parameter modes in one function each.
fn inspect(text: &String) -> usize {
    text.len()
}
fn normalise(text: &mut String) {
    let trimmed = text.trim_end().to_string();
    *text = trimmed;
}
fn consume(text: String) -> usize {
    text.len()
}
let mut text = String::from("  padded  ");
println!("inspect: {}", inspect(&text));
normalise(&mut text);
println!("after normalise: {text:?}");
println!("consume: {}", consume(text));
// println!("{text}");
//
// error[E0382]: borrow of moved value: `text`
//   = note: value moved into `text` here... consumed by `consume`


## 4. The Borrow Rules, and NLL

The two rules, stated once, precisely:

> **Rule 1.** At any moment, a value may have *either* any number of shared borrows (`&T`) *or* exactly one mutable borrow (`&mut T`) — never both.

> **Rule 2.** A borrow may not outlive the value it borrows.

That is the entire model. Everything that feels like "the borrow checker is fussy" is one of these two rules firing on a pattern you did not notice you had written.

### The borrows end at last use, not at end of scope

**Non-Lexical Lifetimes (NLL)** means a borrow's region is determined by where it is *used*, not by where its binding goes out of scope. A borrow that is never used again after line 6 does not conflict with a mutable borrow on line 7, even though both bindings are technically in scope until line 10.


In [ ]:
// 4.1 NLL: the shared borrows are dead by the time the mutable one begins.
let mut text = String::from("base");
let r1 = &text;
let r2 = &text;
println!("reads: {r1} {r2}");
let writer = &mut text;
writer.push('!');
println!("after: {text}");


In [ ]:
// 4.2 Same lines, one use too late: now it is a genuine violation.
//let mut text = String::from("base");
//let r1 = &text;
//let writer = &mut text;
//println!("{r1}");
//writer.push('!');
//
// error[E0502]: cannot borrow `text` as mutable because it is also borrowed
//               as immutable
//  |
//  | let r1 = &text;
//  |          ----- immutable borrow occurs here
//  | let writer = &mut text;
//  |              ^^^^^^^^^ mutable borrow occurs here
//  | println!("{r1}");
//  |             -- immutable borrow later used here


In [ ]:
// 4.3 Successive mutable borrows are fine; overlapping ones are not.
let mut value = 5i32;
let m1 = &mut value;
*m1 += 1;
let m2 = &mut value;
*m2 += 1;
println!("value = {value}");


In [ ]:
// 4.4 Rule 2: a borrow may not outlive its owner.
//fn dangling() -> &String {
//    let local = String::from("temporary");
//    &local
//}
//
// error[E0106]: missing lifetime specifier
//  = help: this function's return type contains a borrowed value, but there
//          is no value for it to be borrowed from
//
// Even with a lifetime annotation, the borrow outlives the owner, so the
// function cannot exist in this shape. Return an owned String instead.


## 5. Slices as Borrowed Views

A **slice** is a view into a contiguous run of elements: `&[T]` for arrays/`Vec`s, `&str` for text. It is a **fat pointer** — pointer plus length, 16 bytes on a 64-bit target — and it is a borrow of someone else's buffer. Slicing (`&data[1..4]`) never copies; it computes a new fat pointer over the same bytes.

| Type | Size | Owns the buffer? | Typical role |
|---|---|---|---|
| `[T; N]` | `N * size_of::<T>()` | yes | fixed-size data, stack-friendly |
| `&[T]` | 16 | no | read-only view, the general parameter |
| `&mut [T]` | 16 | no | in-place edit of someone's buffer |
| `Vec<T>` | 24 | yes | growable storage |


In [ ]:
// 5.1 A slice is a fat pointer over someone else's memory.
let data = [10i32, 20, 30, 40, 50];
let middle: &[i32] = &data[1..4];
println!("slice = {middle:?}, len = {}", middle.len());
println!("sizes: &[i32] = {}, [i32; 5] = {}",
    std::mem::size_of::<&[i32]>(), std::mem::size_of::<[i32; 5]>());


In [ ]:
// 5.2 The coercion chain: array -> &[T; N] -> &[T] at the call site.
fn total(values: &[i32]) -> i32 {
    values.iter().sum()
}
let data = [1i32, 2, 3, 4];
println!("array coerces: {}", total(&data));
let heap = vec![5i32, 6, 7];
println!("Vec coerces: {}", total(&heap));
println!("sub-slice: {}", total(&heap[1..]));


In [ ]:
// 5.3 &mut slices: the callee edits the caller's buffer in place.
fn clip(values: &mut [i32]) {
    for value in values.iter_mut() {
        if *value < 0 {
            *value = 0;
        }
    }
}
let mut readings = vec![3, -1, 4, -15, 9];
clip(&mut readings);
println!("readings = {readings:?}");


### Splitting one buffer into disjoint mutable views

The one-mutable-borrow rule would seem to forbid touching two halves of the same array at once. `split_at_mut` is the standard-library escape hatch — and it is safe, because the two returned slices are *provably disjoint* (the library carries an `unsafe` core so your code does not have to).


In [ ]:
// 5.4 split_at_mut: two disjoint &mut views into one buffer, at the same time.
let mut readings = [3.0f64, 1.0, 4.0, 1.5];
let (left, right) = readings.split_at_mut(2);
left[0] = 30.0;
right[1] *= 2.0;
println!("readings = {readings:?}");


## 6. `Box` and Moving Out

`Box<T>` is the simplest owning pointer: one `T` on the heap, one 8-byte pointer on the stack. It exists for the cases where "put this behind a pointer" is the whole job — recursive types, moving a large value cheaply, holding a trait object.

| Expression | Effect |
|---|---|
| `Box::new(value)` | allocates, moves `value` in |
| `*boxed` | **moves** the `T` out (the box is consumed) |
| `&*boxed`, `&boxed[..]` | borrows through the box |
| `boxed.field` | auto-derefs, borrow or Copy-read |

Reading through a box with `*` in a *value* position moves out of it — which consumes the box. Reading a `Copy` field through the deref copies, exactly as it would without the box. The general principle, worth naming now because it explains a whole family of errors: **you cannot move a non-`Copy` value out of a shared borrow**, and `E0507` is what the compiler says when you try.


In [ ]:
// 6.1 Box is an owning pointer; deref reads through transparently.
let boxed = Box::new(7i32);
println!("boxed = {boxed}, *boxed = {}, size = {}",
    *boxed, std::mem::size_of::<Box<i32>>());


In [ ]:
// 6.2 Moving out of a box consumes it; borrowing through it does not.
let boxed = Box::new(String::from("inside"));
let text = *boxed;
println!("moved out: {text}");
let boxed2 = Box::new(String::from("stays"));
let view: &str = &boxed2;
println!("borrowed through: {view}");
println!("still own boxed2: {boxed2}");


In [ ]:
// 6.3 The rule: no moving a non-Copy value out from behind a shared borrow.
let owned = String::from("payload");
let r = &owned;
// let extracted = *r;
//
// error[E0507]: cannot move out of `*r` which is behind a shared reference
//  |
//  | let extracted = *r;
//  |                 ^^ move occurs because `*r` has type `String`, which
//  |                    does not implement the `Copy` trait
// help: consider removing the dereference here
//  |
//  | let extracted = r;
// println!("{r}");
println!("the value stays put: {r}");


## 7. Structs and Enums: Ownership in Composition

A struct owns its fields; an enum owns its payload. Nothing new is enforced here that was not true of plain variables — but composition is where partial moves and match-by-reference come from, so the details are worth one section.

**Partial move**: you may move one field out of a struct and keep using the `Copy` fields, but the struct as a whole is now partly hollowed out, and the compiler tracks exactly which parts are still valid.

**Match ergonomics**: matching on a reference (`match code { ... }` where `code: &Option<String>`) automatically binds the payload as a reference (`text: &String`). You only write `&` in patterns when you want to match by reference *explicitly* — and matching a non-`Copy` payload by value, out of a reference, is precisely the `E0507` from §6.


In [ ]:
// 7.1 Partial moves hollow out exactly the moved fields.
let position = (String::from("BTC-USD"), 3, 41_250.0);
let (symbol, size, _) = position;
println!("symbol = {symbol}, size = {size}");
// println!("{position:?}");
//
// error[E0382]: borrow of partially moved value: `position`
//   = note: partial move occurs because `position.0` has type `String`,
//           which does not implement the `Copy` trait


In [ ]:
// 7.2 Match on a reference binds references: the vec stays fully owned.
let statuses = vec![Some(String::from("ok")), None, Some(String::from("retry"))];
for status in &statuses {
    match status {
        Some(text) => println!("Some({text})"),
        None => println!("None"),
    }
}
println!("statuses still owned here: {statuses:?}");


In [ ]:
// 7.3 Match by value consumes; match by reference does not.
let status = Some(String::from("consume me"));
match status {
    Some(text) => println!("payload moved out: {text}"),
    None => println!("nothing"),
}
// println!("{status:?}");
// error[E0382]: borrow of moved value: `status`
//   = note: `Option<String>` moved... in the `Some` arm
let reference = Some(String::from("borrow me"));
if let Some(text) = &reference {
    println!("payload borrowed: {text}");
}
println!("still here: {reference:?}");


## 8. Ownership in Function Signatures

Every parameter is a contract about ownership, and the choice is visible to the caller without reading the body:

| Signature | Caller's value afterwards | Callee's obligation |
|---|---|---|
| `fn f(x: T)` | moved away | owns and may consume it |
| `fn f(x: &T)` | intact, read-only during | reads only, may not keep the borrow after return |
| `fn f(x: &mut T)` | intact after return | may mutate, must not leak the borrow |
| `fn f(x: Box<T>)` | moved | owns the heap value |
| `fn f(x: &[T])` / `f(x: &str)` | intact | reads a view, no allocation |

The mirror-image rule for **returns**: a function cannot return a borrow of a local (§4.4), so anything the function *creates* must be returned owned. Together these two halves produce the idiomatic shapes you have already met in chapters 03 and 04 — borrow in, own out:


In [ ]:
// 8.1 Borrow in, own out: the shape of nearly every Rust function.
fn loudest<'a>(inputs: &'a [&'a str]) -> Option<&'a str> {
    inputs.iter().copied().max_by_key(|text| text.len())
}
fn doubles(values: &[i32]) -> Vec<i32> {
    values.iter().map(|v| v * 2).collect()
}
let quotes = ["ask", "bid", "last"];
println!("loudest = {:?}", loudest(&quotes));
println!("doubles = {:?}", doubles(&[1, 2, 3]));
let source = String::from("input");
let doubled = doubles(&source.bytes().map(|b| b as i32).collect::<Vec<_>>());
println!("doubled bytes = {}", doubled.len());


In [ ]:
// 8.2 The same function at three ownership levels.
#[allow(dead_code)] // deliberately not called: calling it would move `text`
fn count_owned(text: String) -> usize {
    text.len()
}
fn count_borrow(text: &str) -> usize {
    text.len()
}
fn count_through_box(text: &Box<String>) -> usize {
    text.len()
}
let text = String::from("count me");
println!("borrowed: {}", count_borrow(&text));
println!("through a box: {}", count_through_box(&Box::new(text.clone())));
println!("borrow again: {}", count_borrow(&text));
// let consumed = count_owned(text);
// println!("{text}");
//
// error[E0382]: borrow of moved value: `text`


## 9. Method Receivers: `&self`, `&mut self`, `self`

A method's receiver is the same choice as a function parameter, spelled with `self`:

| Receiver | Meaning | Use when |
|---|---|---|
| `&self` | borrows, read-only | the overwhelming default |
| `&mut self` | borrows, exclusive | mutation of `self`'s fields |
| `self` | **consumes** the value | builders, converting into another type |
| `Box<Self>`, `Rc<Self>`… | smart-pointer receivers | rare; chapter 13 |

`self` by value is the one that surprises people: a method taking `self` **consumes** the receiver, and the caller cannot use it afterwards. That is exactly what a builder wants — `let app = App::new().with_name("x").with_retries(3);` — and exactly wrong for a value you intend to keep querying.


In [ ]:
// 9.1 The three receivers on one type.
struct Book {
    title: String,
    pages: u32,
}
impl Book {
    fn pages(&self) -> u32 {
        self.pages
    }
    fn rename(&mut self, title: &str) {
        self.title = title.to_string();
    }
    fn into_title(self) -> String {
        self.title
    }
}
let mut book = Book { title: String::from("Draft"), pages: 120 };
book.rename("Ownership");
println!("{} — {} pages", book.title, book.pages());
let title = book.into_title();
println!("extracted: {title}");
// println!("{}", book.pages());
//
// error[E0382]: borrow of moved value: `book`
//   = note: `book` moved due to this method call... `into_title` takes `self`


### Auto-ref: why `text.push('!')` works on a non-`mut` binding... it doesn't

Method calls auto-ref: `s.len()` means `(&s).len()`, and `s.push(x)` means `(&mut s).push(x)`. The receiver choice in the `impl` decides which one is available — which is why a `&self` method can be called through anything, a `&mut self` method needs the binding to be `mut`, and a `self` method consumes whatever the expression evaluates to.

One consequence worth memorising for later chapters: `for` loops, method chains and iterator adaptors are all governed by the same three-way choice (`iter()` / `iter_mut()` / `into_iter()`), and chapter 06 will make that connection explicit.


In [ ]:
// 9.2 Auto-ref in both directions.
let mut text = String::from("grow");
text.push('!');
println!("{text}");
let read_only = String::from("read");
println!("{}", read_only.len());


## 10. Lifetimes and Elision

A **lifetime** is the region of program execution during which a borrow is valid. The annotation `'a` does not *change* how long anything lives; it *names* a relationship so the compiler can check it. Most of the time you never write one, because **elision** — a small set of deterministic rules — fills them in.

The rules, for functions:

1. Each elided input lifetime gets its own distinct parameter (`&str`, `&str` → `'a`, `'b`).
2. If there is exactly one input lifetime, every elided output lifetime is that one.
3. If `&self` or `&mut self` is present, every elided output lifetime is `self`'s.

Rule 2 is why `fn first_word(text: &str) -> &str` is complete as written: one input reference, so the output is tied to it. Rule 3 is why accessors work without annotations.


In [ ]:
// 10.1 Elision in action: one input reference, output tied to it by rule 2.
fn first_word(text: &str) -> &str {
    match text.find(' ') {
        Some(index) => &text[..index],
        None => text,
    }
}
let line = String::from("borrow checker");
let word = first_word(&line);
println!("first word = {word}");
println!("line still owned and intact: {line}");


In [ ]:
// 10.2 Rule 3: methods get their output lifetime from &self for free.
struct Buffer {
    text: String,
}
impl Buffer {
    fn as_str(&self) -> &str {
        &self.text
    }
}
let buffer = Buffer { text: String::from("buffered") };
let view = buffer.as_str();
println!("view = {view}");


In [ ]:
// 10.3 When elision cannot decide: two inputs, one output.
//fn longer(x: &str, y: &str) -> &str {
//    if x.len() > y.len() { x } else { y }
//}
//
// error[E0106]: missing lifetime specifier
//  |
//  | fn longer(x: &str, y: &str) -> &str {
//  |               ----      ----     ^ expected named lifetime parameter
//  |
//  = help: this function's return type contains a borrowed value, but the
//          signature does not say whether it is borrowed from `x` or `y`
// help: consider introducing a named lifetime parameter
//  |
//  | fn longer<'a>(x: &'a str, y: &'a str) -> &'a str {


In [ ]:
// 10.4 The annotation names a relationship; the compiler checks the callers.
fn longer<'a>(x: &'a str, y: &'a str) -> &'a str {
    if x.len() > y.len() { x } else { y }
}
let a = String::from("long");
let result;
{
    let b = String::from("tiny");
    result = longer(&a, &b);
    println!("longer = {result}");
}
// println!("{result}");
//   <- the borrow of `b` would outlive `b` here
//      (error[E0597]: `b` does not live long enough)
println!("a is still alive: {a}");


### Structs that hold borrows need annotations too

A struct holding `&T` borrows from someone, so the borrow's lifetime must be named in the struct definition. This is legal but uncommon — the usual alternative is to store owned data, or to compute views on demand via `&self` methods (rule 3 above).


In [ ]:
// 10.5 A struct with a borrowed field: the annotation is part of the type.
struct Excerpt<'a> {
    text: &'a str,
}
let novel = String::from("the quick brown fox");
let excerpt = Excerpt { text: &novel[4..9] };
println!("excerpt = {}", excerpt.text);
println!("novel intact = {novel}");


## 11. `Option` in Depth

`Option<T>` appeared in chapters 02 and 03 as the null-signal type; with ownership in hand it becomes a **move-aware container**. The payload is either present (`Some(T)`) or absent (`None`), and every method's signature tells you whether it inspects, borrows, mutates or consumes.

| Method | Signature idea | Effect on the `Option` |
|---|---|---|
| `is_some` / `is_none` | `&self` | untouched |
| `as_ref` | `&self -> Option<&T>` | untouched; borrow the payload |
| `as_mut` | `&mut self -> Option<&mut T>` | borrow the payload mutably |
| `take` | `&mut self -> Option<T>` | payload out, `None` left behind |
| `replace` | `&mut self -> Option<T>` | payload out, new one in |
| `unwrap_or`, `unwrap_or_else` | `self` | **consumes** (chapter 01, §10 gotcha) |
| `get_or_insert` | `&mut self -> &mut T` | inserts `None`-case value, yields `&mut T` |
| `map` | `self` | consumes, transforms the payload |
| `?` | in `fn -> Option<_>` | returns early on `None` |

The `take`-then-act pattern is the ownership-native way to update a slot in place: pull the payload out, compute, put something back — without ever cloning.


In [ ]:
// 11.1 as_ref borrows; the original stays put.
let slot = Some(String::from("payload"));
let view = slot.as_ref();
println!("view = {view:?}");
println!("slot untouched = {slot:?}");


In [ ]:
// 11.2 take empties the slot and hands you the payload.
let mut slot = Some(String::from("process me"));
while let Some(task) = slot.take() {
    println!("handling {task:?}");
    if task.ends_with("me") {
        slot = Some(format!("{task} (done)"));
    }
}
println!("slot drained: {slot:?}");


In [ ]:
// 11.3 map consumes; as_ref().map() borrows. Same result, different ownership.
let name = Some(String::from("ann"));
let loud = name.as_ref().map(|text| text.to_uppercase());
println!("borrowed map: {loud:?}, name = {name:?}");
let name2 = Some(String::from("bob"));
let consumed = name2.map(|text| text.len());
println!("consuming map: {consumed:?}");


In [ ]:
// 11.4 get_or_insert: ensure a slot exists, then mutate through &mut T.
let mut settings: Option<String> = None;
let current = settings.get_or_insert_with(|| String::from("default"));
current.push_str(" (tuned)");
println!("settings = {settings:?}");
let again = settings.get_or_insert_with(|| String::from("unused"));
again.push('!');
println!("settings = {settings:?}");


### `Option` is free for references

`Option<&T>` has the same size as `&T`, because a reference is never 0 and the compiler uses 0 as the `None` niche. The same holds for `Option<Box<T>>` and `Option<NonZeroU32>`. It does **not** hold for `Option<i32>` — `0` is a valid `i32` — so that one is 8 bytes, not 4.

| Type | Size | Why |
|---|---|---|
| `&i32` | 8 | pointer |
| `Option<&i32>` | 8 | niche optimisation |
| `Option<i32>` | 8 | 4 for the payload, 1 for the discriminant, padded |
| `Option<Box<u8>>` | 8 | niche optimisation |


In [ ]:
// 11.5 The niche optimisation, measured.
println!("size_of::<&i32>()            = {}", std::mem::size_of::<&i32>());
println!("size_of::<Option<&i32>>()    = {}", std::mem::size_of::<Option<&i32>>());
println!("size_of::<Option<i32>>()     = {}", std::mem::size_of::<Option<i32>>());
println!("size_of::<Option<Box<u8>>>() = {}", std::mem::size_of::<Option<Box<u8>>>());


## 12. The Borrow Checker in Real Loops

Loops are where the borrow rules stop being theory. Three patterns account for nearly all real-world borrow errors, and each has a mechanical fix.

### Pattern 1: iterating by value moves the collection

`for item in items` calls `into_iter()` and **moves** `items`. If the loop body touches `items` at all — even `items.len()` — that is a use-after-move. The fixes, in order of preference: iterate by reference (`for item in &items`), or collect what you need before the loop.


In [ ]:
// 12.1 by-value iteration moves; the compiler's note names into_iter.
let words = vec![String::from("alpha"), String::from("beta")];
for word in words.iter() {
    println!("still owner here: {words:?} contains {word:?}");
}
for word in &words {
    println!("same thing, idiomatic spelling: {word:?}");
}


In [ ]:
// 12.2 The commented-out failure, for reference:
//let mut words = vec![String::from("alpha"), String::from("beta")];
//for word in words {
//    words.push(String::from("gamma"));
//    let _ = word;
//}
//
// error[E0382]: borrow of moved value: `words`
//  |
//  | for word in words {
//  |              ----- `words` moved due to this implicit call to `.into_iter()`
//  |     words.push(String::from("gamma"));
//  |         ^^^^^ value borrowed here after move


### Pattern 2: mutating a collection while iterating it by reference

`for item in &items` holds a shared borrow of `items` for the whole loop. A `push` inside the body wants a mutable borrow — conflict, `E0502`. The fix is to not fight it: collect the new items into a side list and `extend` after the loop, or restructure around indices.


In [ ]:
// 12.3 Collect-then-extend: the standard fix for grow-while-iterating.
let mut words = vec![String::from("alpha"), String::from("beta")];
let mut additions: Vec<String> = Vec::new();
for word in &words {
    if word.len() == 5 {
        additions.push(format!("{word}-x"));
    }
}
words.extend(additions);
println!("{words:?}");


### Pattern 3: removing while iterating a map

Same shape as pattern 2, one level deeper. The borrow-rule-native fix is `retain`: the mutation happens inside the container's own borrow, where the exclusivity is already held.


In [ ]:
// 12.4 retain: mutate a collection in place without fighting the iterator.
let mut scores: std::collections::HashMap<String, i32> = [
    (String::from("alpha"), 3),
    (String::from("beta"), 0),
    (String::from("gamma"), 7),
]
.into_iter()
.collect();
scores.retain(|name, score| {
    if *score == 0 {
        println!("dropping {name}");
        false
    } else {
        true
    }
});
println!("{scores:?}");


## 13. Reading the Compiler: The Five Codes

Five diagnostics account for nearly all borrow errors. Each one names the rule it enforces, points at the two relevant lines, and usually suggests the fix. Reading them is faster than searching for them.

### `E0382` — use of a moved value

```text
error[E0382]: borrow of moved value: `s1`
  |
  | let s1 = String::from("hello");
  |         -- move occurs because `s1` has type `String`, which does not
  |            implement the `Copy` trait
  | let s2 = s1;
  |            -- value moved here
  | println!("{s1}");
  |             ^^ value borrowed here after move
help: consider cloning the value if the performance cost is acceptable
  |
  | let s2 = s1.clone();
```

Cause: use-after-move. Fix: clone (explicit cost), restructure so the move happens last, or work with borrows instead of ownership.

### `E0502` — cannot borrow as mutable because also borrowed as immutable

```text
error[E0502]: cannot borrow `text` as mutable because it is also borrowed
              as immutable
  |
  | let r1 = &text;
  |          ----- immutable borrow occurs here
  | let writer = &mut text;
  |              ^^^^^^^^^ mutable borrow occurs here
  | println!("{r1}");
  |             -- immutable borrow later used here
```

Cause: rule 1 violation — shared and mutable borrows overlap. Fix: reorder so the shared borrows are dead (NLL usually just wants the last read moved up), or scope the borrows with `{}`.

### `E0507` — cannot move out of a shared reference

```text
error[E0507]: cannot move out of `*r` which is behind a shared reference
  |
  | let extracted = *r;
  |                 ^^ move occurs because `*r` has type `String`, which
  |                    does not implement the `Copy` trait
help: consider removing the dereference here
  |
  | let extracted = r;
```

Cause: trying to move a non-`Copy` value out from behind `&`. Fix: clone the value, or take `&T` and only read, or take `T` by value if ownership transfer is the intent.

### `E0499` — more than one mutable borrow

```text
error[E0499]: cannot borrow `x` as mutable more than once at a time
```

Cause: two live `&mut` borrows to the same target. Fix: let the first die before the second is born (NLL often makes this automatic once uses are reordered), or split the data with `split_at_mut` when the two borrows are genuinely disjoint.

### `E0106` — missing lifetime specifier

```text
error[E0106]: missing lifetime specifier
  |
  | fn longer(x: &str, y: &str) -> &str {
  |               ----      ----     ^ expected named lifetime parameter
  |
  = help: this function's return type contains a borrowed value, but the
          signature does not say whether it is borrowed from `x` or `y`
help: consider introducing a named lifetime parameter
  |
  | fn longer<'a>(x: &'a str, y: &'a str) -> &'a str {
```

Cause: elision cannot decide which input an output borrow comes from. Fix: name the relationship with `'a` — or, more often, return an owned value (`String`, `Vec<T>`) because the function actually produces new data.

### Honorable mentions

| Code | One-line cause | One-line fix |
|---|---|---|
| `E0505` | move while borrowed | end the borrow before the move |
| `E0597` | borrowed value does not live long enough | the owner must outlive the borrow: restructure scopes |
| `E0506` | cannot assign to a value because it is borrowed | drop the borrow before assigning |
| `E0596` | cannot borrow as mutable, not declared `mut` | the *binding* needs `mut` |
| `E0382` (partial) | `borrow of partially moved value` | the moved field is gone; use the field, not the struct |

### The pattern in all five

Every one of these is a disagreement about ownership that the compiler refused to guess at. The `help:` lines usually contain the mechanical fix; the two-line story (where the first borrow started, where the second one clashed) contains the reason. Reading both is the habit — and when neither is clear, renaming the bindings to say what they borrow often reveals the problem before the compiler does.


In [ ]:
// 13.1 All five codes are comments; the legal spellings run here.
let s1 = String::from("hello");
let s2 = s1.clone();
println!("E0382 avoided by clone: {s1} / {s2}");

let mut text = String::from("base");
let read = &text;
println!("E0502 avoided by NLL: {read}");
let writer = &mut text;
writer.push('!');
println!("...then exclusive: {text}");

let owned = String::from("payload");
let r = &owned;
println!("E0507 avoided by reading: {r}");

let mut value = 5;
let m1 = &mut value;
*m1 += 1;
let m2 = &mut value;
*m2 += 1;
println!("E0499 avoided by NLL: {value}");

fn leading_word(text: &str) -> &str {
    text.split(' ').next().unwrap_or(text)
}
println!("E0106 avoided by elision: {:?}", leading_word("one two"));


## 14. Performance Notes for Low-Latency Code

Ownership is not just correctness machinery; it is the performance model. The moves you make explicit are the copies other languages hide, and the borrows are the aliasing information optimisers usually have to guess at.

| Cost | Operation |
|---|---|
| Free (pointer bump) | move of any owning type (`String`, `Vec`, `Box`) |
| Free | `Copy` of small types (the compiler may not even emit it) |
| Free | creating a borrow |
| O(n) | `clone()` of a `String`/`Vec` — the explicit choice |
| Heap alloc | `String::from`, `vec![]`, `Box::new` |

Three habits follow directly:

1. **Prefer borrows in hot paths.** A `&str`/`&[T]` parameter allocates nothing; a `String`/`Vec<T>` parameter forces the caller to have allocated, or to clone.
2. **Move, do not clone, when transferring.** `let b = a;` on an owning type is a register copy. `a.clone()` is a heap allocation plus a memcpy. If you find yourself cloning to appease the compiler, look for a borrow-shaped restructure first — then clone deliberately if it is genuinely the right trade.
3. **Borrow aggressively locally.** NLL means short borrows almost never conflict. The scoped-borrow idiom (`let value = { let r = &data; r.compute() };`) reads awkwardly but keeps exclusivity windows minimal, which matters both for the compiler and for anyone reading the code.

The inverse warning: do not contort designs to avoid one clone on a cold path. The borrow checker's rejects are cheap to fix; the contortions are not.


In [ ]:
// 14.1 Moves are pointer-bumps; measure the difference from clone.
let payload = String::from("a payload of some length");
let moved = payload;
println!("move: len {}", moved.len());
let payload2 = String::from("a payload of some length");
let cloned = payload2.clone();
println!("clone: len {} (two allocations exist)", cloned.len());
let borrowed = &payload2;
println!("borrow: len {} (no allocation at all)", borrowed.len());


## 15. Mental Model and Habits

### The picture

Everything in this chapter is one diagram:

```text
owner (stack frame or container)
  │
  ├── owns the value ────────────── frees it at end of scope
  │
  ├── &T    ── shared borrow: many, read-only ──┐
  │                                             ├── borrow rules (§4)
  └── &mut T ── exclusive borrow: one ──────────┘
```

A move rewires the owner arrow. A borrow draws a temporary arrow that the compiler checks for (a) exclusivity conflicts and (b) outliving the owner. That is all; there is no runtime component anywhere in the diagram.

### Habits that prevent most fights

1. **Read the signature first.** `T` / `&T` / `&mut T` tells you who owns what after the call. Most "borrow checker" surprises are signature surprises.
2. **Order operations by use.** Write reads before writes, and let the last read happen before the first write. NLL then does the rest.
3. **Scope borrows deliberately** when a long function mixes reads and writes: `{ let r = &x; use(r) }` then mutate. The braces are documentation of intent.
4. **Clone at the boundary, not in the loop.** One clone where the data enters your function is a design; clones inside a hot loop are a bug.
5. **When stuck, shrink the problem**: comment out the failing line, get the rest compiling, and reintroduce it. The minimal failing program is the one whose diagnostic finally makes sense.
6. **Do not use `unsafe` to "skip" the borrow checker.** It does not skip anything: it moves the checking onto you, at every call site, forever, with no compiler help. This series stays in safe Rust.

### What the borrow checker is *not*

It is not a garbage collector (no runtime, no pauses). It is not reference counting (`Rc` exists, chapter 13, and is opt-in). It is not a style enforcer (an accepted program is not "better", just provably alias-free). It is a compile-time proof that every use of every value is legal — which is why Rust programs that compile tend to simply work.


## 16. Cheat Sheet

### The two rules

| Rule | Statement |
|---|---|
| Moves | every non-`Copy` transfer moves; the old binding is dead |
| Borrows | many `&T` xor one `&mut T`; never outlive the owner |

### The three parameter modes

| Parameter | Caller keeps value | Callee can mutate | Use for |
|---|---|---|---|
| `T` | no | owns | consuming, builders |
| `&T` / `&str` / `&[T]` | yes | no | reads |
| `&mut T` / `&mut [T]` | yes (after) | yes | in-place edits |

### `Option` ownership quick reference

| Need | Write |
|---|---|
| look at payload | `.as_ref()` |
| mutate payload | `.as_mut()` |
| take payload out, leave `None` | `.take()` |
| swap in a new payload | `.replace(new)` |
| get `&mut T`, inserting if absent | `.get_or_insert_with(|| ...)` |
| transform, consuming | `.map(\|x\| ...)` |
| transform, borrowing | `.as_ref().map(\|x\| ...)` |
| early return on `None` | `?` (in `fn -> Option<_>`) |

### The five diagnostics

| Code | Cause | First fix to try |
|---|---|---|
| `E0382` | use after move | clone, or reorder so the move is last |
| `E0502` | shared + mutable overlap | move the last read above the write |
| `E0507` | move out of a shared borrow | clone the value, or read instead of move |
| `E0499` | two live `&mut` | reorder uses; `split_at_mut` for disjoint halves |
| `E0106` | elision can't pick the output lifetime | name `'a`, or return owned |

### Slices

| Need | Write |
|---|---|
| read a run of elements | `&data[a..b]` → `&[T]` |
| edit in place | `&mut data[a..b]` → `&mut [T]` |
| two disjoint edits | `data.split_at_mut(i)` |
| generic "any array or Vec" parameter | `fn f(items: &[T])` |

### The one habit

When a borrow error appears, **say the two-line story out loud**: where did the first borrow start, and where does the second one clash. If you cannot tell the story from the diagnostic, the problem is not the borrow checker — it is a design that has not yet decided who owns the value.


### What Comes Next

Chapter 05 closes the language core that chapter 01 opened: values now have owners, borrows have rules, and the five diagnostics are readable. Everything ahead builds on it directly.

| Chapter | Title | What it settles |
|---|---|---|
| **06** | **Native Collections** | `Vec`, `HashMap`, `VecDeque`, `BTreeMap` — containers whose iterator methods are the borrow rules of this chapter in daily clothing |
| 07 | I/O and Files | `std::fs`, readers and writers, and error propagation with `Result` |
| 10 | Structs, OOP and Traits | the full story of `impl`, traits, and the `Display`/`Debug` machinery used throughout |

**Carry forward.** You should now be able to do these without looking anything up:

- say, for any binding, who owns the value and who is borrowing it
- choose `T` vs `&T` vs `&mut T` in a signature, and predict the caller's experience
- fix `E0382` by reordering or cloning *deliberately*, and `E0502` by reordering reads and writes
- write a slice-taking function that accepts arrays and `Vec`s alike
- reach for `as_ref`, `take`, `get_or_insert` instead of cloning an `Option`'s payload

If any of those still needs looking up, §16 is the one-page answer, and re-running the matching section here is the check.
